# Матричная факторизация на практике: `implicit`

Ноутбук к лекции 3 "Матричная факторизация: SVD, ALS, iALS, BPR-MF".

Библиотека **`implicit`** (benfred/implicit) --- лёгкая, быстрая (Cython/C++ под капотом), без Spark и JVM,
реализует три модели для implicit feedback, которые мы разбирали на лекции:

- **ALS / iALS** --- point-wise, confidence-веса, замкнутая форма на каждом шаге
- **BPR** --- pairwise, ранжирующий loss, обучение по тройкам $(u,i,j)$
- **LMF** (Logistic Matrix Factorization) --- point-wise, но с логистическим loss вместо квадратичного

Метрики, как и раньше, считаем через `replay.metrics` --- он не требует Spark и одинаково хорошо
работает что с игрушечными словарями, что с реальными данными.


## Установка библиотек

In [ ]:
# Если библиотеки ещё не установлены, раскомментируйте:
# !pip install implicit replay-rec scipy pandas numpy matplotlib


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

np.random.seed(42)


## 1. Синтетический датасет

Как и в предыдущем ноутбуке: у пользователей и айтемов есть скрытые векторы предпочтений,
поверх них генерируем implicit-взаимодействия, плюс добавляем перекос по популярности айтемов
(длинный хвост) --- иначе Popularity-бейзлайн и confidence-веса ALS не будет на чём показать себя.

In [ ]:
n_users = 800
n_items = 400
latent_dim = 12

user_vecs = np.random.normal(size=(n_users, latent_dim))
item_vecs = np.random.normal(size=(n_items, latent_dim))
item_popularity_bias = np.random.pareto(a=2.0, size=n_items)

scores = user_vecs @ item_vecs.T + item_popularity_bias[None, :]
probs = 1 / (1 + np.exp(-scores / 3))

# implicit-сигнал: не просто 0/1, а "сила" взаимодействия (число кликов/просмотров)
raw_signal = np.random.poisson(lam=np.clip(probs * 4, 0, None))
interactions = raw_signal.astype(np.float32)

print("Средняя плотность матрицы:", (interactions > 0).mean())
print("Взаимодействий на пользователя (в среднем):", (interactions > 0).sum(axis=1).mean())


### Train/test split

Для каждого пользователя откладываем ~20% взаимодействий в тест.

In [ ]:
train_mask = np.ones_like(interactions)
test_items = {}

for u in range(n_users):
    liked = np.where(interactions[u] > 0)[0]
    if len(liked) < 2:
        continue
    n_test = max(1, int(len(liked) * 0.2))
    test_idx = np.random.choice(liked, size=n_test, replace=False)
    test_items[u] = test_idx.tolist()
    train_mask[u, test_idx] = 0

train_matrix = interactions * train_mask
train_csr = csr_matrix(train_matrix)

ground_truth = {u: items for u, items in test_items.items()}
train_gt = {u: np.where(train_matrix[u] > 0)[0].tolist() for u in range(n_users)}

print("Пользователей с тестовыми айтемами:", len(ground_truth))


## 2. ALS / iALS

`implicit.als.AlternatingLeastSquares` --- реализация именно implicit-версии (iALS с confidence-весами).

Confidence считается по формуле с лекции: $c_{ui} = 1 + \alpha \cdot r_{ui}$. Параметр `alpha` задаёт
силу доверия к частым взаимодействиям --- в `implicit` вход уже ожидается как "сырые" значения $r_{ui}$,
преобразование в confidence модель делает сама.

In [ ]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
    factors=32,
    regularization=0.05,
    alpha=20.0,       # сила confidence, как альфа в c_ui = 1 + alpha * r_ui
    iterations=15,
    random_state=42,
)
als_model.fit(train_csr)


Получаем рекомендации для всех пользователей сразу (батчево):

In [ ]:
def get_recommendations(model, train_csr, k=10):
    user_ids = np.arange(train_csr.shape[0])
    ids, scores = model.recommend(
        user_ids, train_csr[user_ids], N=k, filter_already_liked_items=True
    )
    recs = {}
    for row, u in enumerate(user_ids):
        recs[u] = list(zip(ids[row].tolist(), scores[row].tolist()))
    return recs

als_recs = get_recommendations(als_model, train_csr, k=10)
als_recs[0][:5]


## 3. BPR

`implicit.bpr.BayesianPersonalizedRanking` --- pairwise-обучение по тройкам $(u,i,j)$.

Параметр `verify_negative_samples` (по умолчанию `True`) включает проверку: если случайно
засэмплированный "негатив" $j$ на самом деле есть в истории пользователя --- он пересэмплируется.
Сравним модель с включённой и выключенной проверкой.

In [ ]:
from implicit.bpr import BayesianPersonalizedRanking

bpr_model = BayesianPersonalizedRanking(
    factors=32,
    learning_rate=0.05,
    regularization=0.01,
    iterations=100,
    verify_negative_samples=True,
    random_state=42,
)
bpr_model.fit(train_csr)
bpr_recs = get_recommendations(bpr_model, train_csr, k=10)


In [ ]:
bpr_model_noverify = BayesianPersonalizedRanking(
    factors=32,
    learning_rate=0.05,
    regularization=0.01,
    iterations=100,
    verify_negative_samples=False,
    random_state=42,
)
bpr_model_noverify.fit(train_csr)
bpr_noverify_recs = get_recommendations(bpr_model_noverify, train_csr, k=10)


## 4. Logistic Matrix Factorization (LMF)

`implicit.lmf.LogisticMatrixFactorization` --- point-wise, но с логистическим loss: моделируем
вероятность $P(p_{ui}=1)$ через сигмоиду, а не квадратичную ошибку. Параметр `neg_prop` задаёт,
сколько негативов сэмплировать на каждый позитив.

In [ ]:
from implicit.lmf import LogisticMatrixFactorization

lmf_model = LogisticMatrixFactorization(
    factors=32,
    learning_rate=1.0,
    regularization=0.6,
    iterations=30,
    neg_prop=30,
    random_state=42,
)
lmf_model.fit(train_csr)
lmf_recs = get_recommendations(lmf_model, train_csr, k=10)


## 5. Item-based KNN --- бейзлайн из прошлой лекции

Для контекста добавим ItemKNN (cosine) из `implicit.nearest_neighbours` --- бейзлайн без градиентного
обучения, с которым сравнивали MF на прошлой лекции.

In [ ]:
from implicit.nearest_neighbours import CosineRecommender

knn_model = CosineRecommender(K=20)
knn_model.fit(train_csr)
knn_recs = get_recommendations(knn_model, train_csr, k=10)


## 6. Сравнение через `replay.metrics`

Прогоняем все модели через `OfflineMetrics` --- как и в ноутбуке к лекции 2.

In [ ]:
from replay.metrics import Precision, Recall, MAP, NDCG, Coverage, Novelty, OfflineMetrics

metrics = [
    Precision(10),
    Recall(10),
    MAP(10),
    NDCG(10),
    Coverage(10),
    Novelty(10),
]

all_recs = {
    "ItemKNN (cosine)": knn_recs,
    "ALS / iALS": als_recs,
    "LMF": lmf_recs,
    "BPR (verify=True)": bpr_recs,
    "BPR (verify=False)": bpr_noverify_recs,
}

rows = {}
for name, recs in all_recs.items():
    rows[name] = OfflineMetrics(metrics)(recs, ground_truth, train_gt)

pd.DataFrame(rows).T


### Что стоит увидеть

- **ALS/iALS** обычно точнее ItemKNN за счёт латентного пространства и confidence-весов
- **BPR** чаще всего выигрывает по ranking-метрикам (MAP, NDCG) --- он их напрямую оптимизирует
- **`verify_negative_samples=True`** должен давать небольшой, но заметный прирост точности
  относительно `False` --- за счёт исключения ложных негативов, ценой более долгого обучения
- **LMF** --- зачастую где-то между ALS и BPR по качеству: point-wise, но с более гибким loss, чем квадратичный
- **Coverage/Novelty** у MF-моделей обычно выше, чем у ItemKNN --- латентное пространство обобщает
  на less obvious сочетания пользователь-айтем


In [ ]:
from replay.metrics import Experiment

exp = Experiment(metrics, ground_truth, train_gt)
for name, recs in all_recs.items():
    exp.add_result(name, recs)

exp.compare("ItemKNN (cosine)")


## 7. Подбор гиперпараметров: `alpha` в ALS

Быстро посмотрим, как сила confidence влияет на качество --- воспроизведём интуицию с лекции:
слишком маленький `alpha` --- модель слабо доверяет частым взаимодействиям, слишком большой ---
переобучается на них.

In [ ]:
alpha_values = [1, 5, 20, 50, 100]
alpha_results = {}

for alpha in alpha_values:
    model = AlternatingLeastSquares(factors=32, regularization=0.05, alpha=alpha,
                                     iterations=15, random_state=42)
    model.fit(train_csr)
    recs = get_recommendations(model, train_csr, k=10)
    alpha_results[alpha] = OfflineMetrics([NDCG(10), Precision(10)])(recs, ground_truth, train_gt)

pd.DataFrame(alpha_results).T


## Итоги

- `implicit` даёт три модели матричной факторизации из лекции (ALS/iALS, BPR, LMF) буквально
  в несколько строк, без Spark и JVM --- удобно для прототипирования и небольших датасетов
- `verify_negative_samples` в BPR --- пример того, как детали негативного сэмплинга влияют на качество
- Метрики единообразно считаются через `replay.metrics` независимо от того, чем обучена модель
- **WARP-loss** (обсуждали на лекции как более точный вариант pairwise-обучения) в `implicit` не реализован ---
  для него нужен `LightFM` (`loss='warp'`); при желании можно поставить и сравнить отдельно
